In [9]:
#Aggregator Function
import tensorflow as tf
import numpy as np

# Define the aggregation function
def federated_aggregation(model_files, output_file):
    """
    Aggregates weights from multiple models and creates a global model.

    Parameters:
    - model_files: List of file paths to the client models.
    - output_file: File path to save the aggregated global model.

    Returns:
    - global_model: The aggregated global model.
    """
    # Load all client models
    client_models = [tf.keras.models.load_model(file) for file in model_files]
    
    # Extract weights from all client models
    client_weights = [model.get_weights() for model in client_models]
    
    # Ensure all models have the same structure
    for i in range(1, len(client_weights)):
        if len(client_weights[0]) != len(client_weights[i]):
            raise ValueError("Model architectures do not match across clients.")
    
    # Perform federated averaging of weights
    aggregated_weights = []
    for weights in zip(*client_weights):
        aggregated_weights.append(np.mean(weights, axis=0))
    
    # Create a new model with the same architecture as the clients
    global_model = tf.keras.models.clone_model(client_models[0])
    global_model.set_weights(aggregated_weights)
    
    # Save the global model
    global_model.save(output_file)
    print(f"Global model saved as {output_file}.")
    return global_model

# Paths to the models
model_files = ['./sys1.keras', './sys2.keras']
output_file = './global_model.keras'

# Aggregate models
global_model = federated_aggregation(model_files, output_file)

Global model saved as ./global_model.keras.


In [20]:
from keras.models import load_model
from keras.preprocessing.image import ImageDataGenerator
import numpy as np

# Load the saved model
model = load_model('./global_model.keras')

# Data directory
test_data_dir = './Weather_Test'

# Create ImageDataGenerator for test data
test_data_generator = ImageDataGenerator(rescale=1./255)

# Create test data generator
test_gen = test_data_generator.flow_from_directory(
    test_data_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False  # Do not shuffle for evaluation
)

# Make predictions
predictions = model.predict(test_gen)

# Get class labels from the generator
class_indices = test_gen.class_indices
# Reverse the class_indices dictionary to get a mapping from index to label
labels = {v: k for k, v in class_indices.items()}

# Convert predictions to class indices
predicted_class_indices = np.argmax(predictions, axis=1)

# Map the predicted class indices to class labels
predicted_labels = [labels[k] for k in predicted_class_indices]

# Get the filenames
filenames = test_gen.filenames

# Print the predictions with filenames
for filename, predicted_label in zip(filenames, predicted_labels):
    print(f'File: {filename}, Predicted label: {predicted_label}')


Found 40 images belonging to 4 classes.
2/2 [==============================] - 1s 56ms/step
File: dew\2208.jpg, Predicted label: dew
File: dew\2209.jpg, Predicted label: rain
File: dew\2210.jpg, Predicted label: rain
File: dew\2211.jpg, Predicted label: dew
File: dew\2212.jpg, Predicted label: rain
File: dew\2213.jpg, Predicted label: dew
File: dew\2214.jpg, Predicted label: dew
File: dew\2215.jpg, Predicted label: dew
File: dew\2216.jpg, Predicted label: rain
File: dew\2217.jpg, Predicted label: rain
File: frost\3600.jpg, Predicted label: rain
File: frost\3601.jpg, Predicted label: rain
File: frost\3602.jpg, Predicted label: rain
File: frost\3603.jpg, Predicted label: rain
File: frost\3604.jpg, Predicted label: rain
File: frost\3605.jpg, Predicted label: rain
File: frost\3606.jpg, Predicted label: rain
File: frost\3607.jpg, Predicted label: rain
File: frost\3608.jpg, Predicted label: rain
File: frost\3609.jpg, Predicted label: rain
File: rain\102.jpg, Predicted label: rain
File: rain\

In [14]:
model1 = tf.keras.models.load_model('./sys1.keras')
model2 = tf.keras.models.load_model('./sys2.keras')
global_model = tf.keras.models.load_model('./global_model.keras')

print("Model 1 weight sizes:", [w.shape for w in model1.get_weights()])
print("Model 2 weight sizes:", [w.shape for w in model2.get_weights()])
print("Global Model weight sizes:", [w.shape for w in global_model.get_weights()])


Model 1 weight sizes: [(3, 3, 3, 64), (64,), (3, 3, 64, 64), (64,), (61504, 32), (32,), (32, 4), (4,)]
Model 2 weight sizes: [(3, 3, 3, 64), (64,), (3, 3, 64, 64), (64,), (61504, 32), (32,), (32, 4), (4,)]
Global Model weight sizes: [(3, 3, 3, 64), (64,), (3, 3, 64, 64), (64,), (61504, 32), (32,), (32, 4), (4,)]


In [15]:
model1_weights = model1.get_weights()
model2_weights = model2.get_weights()
global_weights = global_model.get_weights()

for layer1, layer2, global_layer in zip(model1_weights, model2_weights, global_weights):
    print(f"Layer comparison:")
    print(f"Max layer 1: {np.max(layer1)}, Min layer 1: {np.min(layer1)}")
    print(f"Max layer 2: {np.max(layer2)}, Min layer 2: {np.min(layer2)}")
    print(f"Max global: {np.max(global_layer)}, Min global: {np.min(global_layer)}")


Layer comparison:
Max layer 1: 0.10653631389141083, Min layer 1: -0.10585735738277435
Max layer 2: 0.11074496060609818, Min layer 2: -0.10758931189775467
Max global: 0.09918591380119324, Min global: -0.09581537544727325
Layer comparison:
Max layer 1: 0.011930475942790508, Min layer 1: -0.007566134911030531
Max layer 2: 0.014313147403299809, Min layer 2: -0.00989814568310976
Max global: 0.009796504862606525, Min global: -0.007886235602200031
Layer comparison:
Max layer 1: 0.0852188989520073, Min layer 1: -0.08313571661710739
Max layer 2: 0.08576207607984543, Min layer 2: -0.08472917973995209
Max global: 0.07997947931289673, Min global: -0.07668411731719971
Layer comparison:
Max layer 1: 0.011573088355362415, Min layer 1: -0.006156958639621735
Max layer 2: 0.010963378474116325, Min layer 2: -0.008357382379472256
Max global: 0.010427254252135754, Min global: -0.007115453016012907
Layer comparison:
Max layer 1: 0.023665884509682655, Min layer 1: -0.022366974502801895
Max layer 2: 0.0241099

In [18]:
#Federated Averaging
import numpy as np
from keras.models import load_model


# Get the model weights from sys1 and sys2
weights1 = model1.get_weights()
weights2 = model2.get_weights()

# Assume N1 and N2 are the number of samples each client has (you should replace them with actual numbers)
N1 = 10  # Example: Number of samples for sys1
N2 = 10  # Example: Number of samples for sys2

# Compute total number of samples
total_samples = N1 + N2

# Compute the weighted average of the model weights
weights_global = []
for w1, w2 in zip(weights1, weights2):
    weighted_w = (w1 * N1 + w2 * N2) / total_samples
    weights_global.append(weighted_w)

# Create a global model and set the aggregated weights
global_model = load_model('./sys1.keras')  # Using sys1's architecture as global model
global_model.set_weights(weights_global)

# Now you can use the global model for prediction or further training
global_model.save('./global_model.keras')
